# 第 11 章　日時と時間の変化

一日の貸出件数がほぼ同じ二日なら，自転車の貸出場所を巡回する担当者も，同じ時間帯に多く配置すればよいでしょうか。このノートでは，日別の合計だけでは分からない，利用が集中する時間帯を調べます。

講義では 6 月 7 日と 9 日を比べた後，土日・祝日とそれ以外の日へ対象を広げます。実習では別の三つの水曜日を選び，朝と夜で最も貸出が多い日が変わるかを調べます。最後に，どの日のどの時間帯の利用に備えるかを，件数と図を使って説明します。

前半は，章本文の説明に合わせて，上から順にセルを実行します。コードの前に何を調べるかを読み，実行後は表や図のどの値が問いへの答えになるかを確かめてください。後半の約 20 分の実習では，比べる対象や条件を自分で選び，結果を短い文章にします。前半のコードは実行できる見本なので，作図の処理をすべて書き写す必要はありません。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。CSV は，手元にファイルがなければ公開 URL から読み込みます。ローカルでは，配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置いて実行できます。

まず，時間別の貸出を `hourly` に読み込みます。先頭の三行で，日付・時間帯と，それぞれの貸出件数を見ておきます。

In [ ]:
from pathlib import Path

import pandas as pd

file_name = "bike-sharing-hourly.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

hourly = pd.read_csv(csv_path)
hourly[["dteday", "hr", "cnt", "workingday"]].head(3)

## 本文と一緒に実行

### 1. 日合計が近い二日を比べる

本文に登場した 2012 年 6 月 7 日と 9 日は，一日の貸出合計がほぼ同じでした。朝の担当者にこの情報だけを伝えてよいか，まず 8 時台の件数を比べます。

次のセルでは，日付を文字列から日時型に変換し，二日分の記録を選びます。選んだ行を日付ごとに合計したあと，別のセルで時間帯ごとの折れ線を描きます。図では，同じ横位置にある二日の点を比べてください。

In [ ]:
print(type(hourly["dteday"].iloc[0]).__name__)
hourly["dteday"] = pd.to_datetime(hourly["dteday"], format="%Y-%m-%d")
print(type(hourly["dteday"].iloc[0]).__name__)
thursday = pd.Timestamp("2012-06-07")
saturday = pd.Timestamp("2012-06-09")
sample_hours = hourly.loc[hourly["dteday"].isin([thursday, saturday])]
display(sample_hours.groupby("dteday")["cnt"].sum())

In [ ]:
import matplotlib.pyplot as plt

weekday_color, weekend_color = "#195b8a", "#a7421f"
fig, ax = plt.subplots(figsize=(7.2, 3.7), layout="constrained")
for date_text, color, marker, style, label in [
    ("2012-06-07", weekday_color, "s", "-", "Jun 7 (Thu)"),
    ("2012-06-09", weekend_color, "o", "--", "Jun 9 (Sat)"),
]:
    values = sample_hours.loc[sample_hours["dteday"] == date_text].sort_values("hr")
    ax.plot(values["hr"], values["cnt"], color=color, marker=marker,
            linestyle=style, markersize=3.4, label=label)
for hour in [8, 13, 17]:
    ax.axvline(hour, color="#9ba6af", linestyle=":", linewidth=0.8, zorder=0)
ax.set(xlabel="Clock hour", ylabel="Rentals per hour", xlim=(-0.3, 23.3),
       ylim=(0, 960), xticks=[0, 4, 8, 13, 17, 23])
ax.legend(frameon=False, loc="upper left")
plt.show()

日合計は 7,494 件と 7,498 件です。一方，8 時台と 13 時台で二日の点の高さを比べると，貸出が多い日は同じでしょうか。それぞれの時間帯について，どちらの日が何件多いかを説明してください。

本文の「二日を比べる」と同じく，一日全体の利用が近いことと，一日の中で利用が多い時間帯が同じことを区別します。

### 2. 日付をまたぐ一時間

時間帯の記録を正しい順序で扱うには，時刻だけでなく日付も必要です。6 月 7 日の 23 時台と翌日の 0 時台を，`hr` の数字だけで並べると何が分からなくなるでしょうか。

次のセルでは，日付にその日の 0 時からの時間数を加えて `timestamp` を作ります。表示された二行で，日付は変わっても時間は順に進んでいることと，貸出件数がどちらの日の合計に入るかを確かめます。

In [ ]:
hourly["timestamp"] = hourly["dteday"] + pd.to_timedelta(hourly["hr"], unit="h")
hourly = hourly.sort_values("timestamp")
midnight = hourly["timestamp"].between("2012-06-07 23:00", "2012-06-08 00:00")
hourly.loc[midnight, ["dteday", "hr", "timestamp", "cnt"]]

### 3. 貸出件数を足すか，時間帯の行を数えるか

一日分の合計を読むときには，何時間分の記録を足したのかも知る必要があります。`sum()` で貸出件数を合計し，`size()` で時間帯の行数を数えて，二つを別の列に並べます。10 月 29 日の「1」と「22」は，それぞれ何を数えた値でしょうか。

続くセルでは，6 月 7 日を 6 時間ごとに分けて小計を求めます。小計を全部足すと日合計へ戻ることと，その小計だけから各時間帯の混み方まで分かるかを考えます。

In [ ]:
by_date = hourly.groupby("dteday")
rentals_per_day = by_date["cnt"].sum()
rows_per_day = by_date.size()

check_days = pd.DataFrame({
    "hours_in_table": rows_per_day,
    "rentals": rentals_per_day,
})
check_days.loc[[thursday, saturday, pd.Timestamp("2012-10-29")]]

In [ ]:
thursday_hours = hourly.loc[hourly["dteday"] == thursday]
time_block = thursday_hours["hr"] // 6
block_totals = thursday_hours.groupby(time_block)["cnt"].sum()
print(block_totals.to_list(), block_totals.sum())
block_totals

6 月 7 日は 24 行の貸出件数を足すと 7,494 件になります。6 時間ごとの小計を足しても，同じ日合計になります。ただし，小計には貸出の多い時間と少ない時間が一緒に含まれます。8 時台や 17 時台の利用を個別に伝えるなら，最初の時間別の図へ戻ります。

### 4. 同じ時刻の平均を比べる

二日で見えた朝と昼の違いは，その二日だけの特徴でしょうか。それとも，土日・祝日とそれ以外の日の利用の仕方の違いでしょうか。後者の説明なら，複数日の同じ時間帯を平均しても，朝と昼で二群の大小が入れ替わりそうです。平均の図を描く前に，どちらの説明を予想するかと，確かめる時間帯を言葉にしてください。

まず 0〜23 時台の記録がそろう日を選びます。これにより，8 時台と 13 時台など，異なる時間帯の平均を同じ日付の集まりから作れます。`workingday` は土日・祝日なら `0`，それ以外なら `1` です。表示される各群の日数を確認してから，平均の折れ線を描きます。

In [ ]:
rows_per_day = hourly.groupby("dteday").size()
complete_dates = rows_per_day.index[rows_per_day == 24]
complete_hours = hourly.loc[hourly["dteday"].isin(complete_dates)]
print("Days with 24 records:", len(complete_dates))
print(complete_hours.groupby("workingday")["dteday"].nunique())
same_hour = complete_hours.groupby(["workingday", "hr"])["cnt"]
mean_by_hour = same_hour.mean()
mean_by_hour.loc[[(0, 8), (1, 8), (0, 13), (1, 13)]].round(2)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.2, 3.7), layout="constrained")
ax.plot(mean_by_hour.loc[1].index, mean_by_hour.loc[1], color="#195b8a",
        marker="s", markersize=3, label="Weekday, excluding holidays")
ax.plot(mean_by_hour.loc[0].index, mean_by_hour.loc[0], color="#a7421f",
        marker="o", markersize=3, linestyle="--", label="Weekend or holiday")
for hour in [8, 13, 17]:
    ax.axvline(hour, color="#9ba6af", linestyle=":", linewidth=0.8, zorder=0)
ax.set(xlabel="Clock hour", ylabel="Mean rentals per hour", xlim=(-0.3, 23.3),
       ylim=(0, 630), xticks=[0, 4, 8, 13, 17, 23])
ax.legend(frameon=True, facecolor="white", edgecolor="none", framealpha=1, loc="upper left")
plt.show()

選んだのは 655 日です。土日・祝日の各点は 212 日，それ以外の各点は 443 日の件数を平均しています。朝・昼・夕方を比べ，同じ群の平均が常に高いわけではないことを，図の時間帯を挙げて説明してください。

比較前の二つの説明に戻り，平均の図がどちらに合うかを，朝と昼の点を使って説明してください。違いが平均にも見えたことと，どの一日にも同じ形が現れることは別です。

この図の一点は，ある一日の実際の件数ではありません。普段どの時間帯に利用が集中するかを考える材料と，個別の日に何件あったかを区別して読みます。

### 5. 暦にあるのに表にない日

利用が少ない時間を探すとき，行がないことを貸出 0 件と考えてよいでしょうか。10 月 27 日〜11 月 2 日から 8〜18 時台を選び，日付ごとに貸出件数を足します。7 日間を指定したのに，集計結果が何行になったかを確かめます。

次に，`pd.date_range()` で期間内の全日付を用意し，`reindex()` で集計結果をその並びに合わせます。値がない日付も含めて描いた折れ線と，集計結果に現れた日付だけを描いた折れ線を比べてください。

In [ ]:
event_start = "2012-10-27"
event_end = "2012-11-02"
event_hours = hourly.loc[hourly["dteday"].between(event_start, event_end)]
daytime = event_hours.loc[event_hours["hr"].between(8, 18)]
observed_daytime = daytime.groupby("dteday")["cnt"].sum()
observed_daytime

In [ ]:
event_calendar = pd.date_range(event_start, event_end, freq="D")
daytime_on_calendar = observed_daytime.reindex(event_calendar)
daytime_on_calendar

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, sharex=True, sharey=True, figsize=(7.2, 5), layout="constrained")
for ax, values, title in zip(
    axes, [observed_daytime, daytime_on_calendar],
    ["Dates present after grouping", "All calendar dates"],
):
    ax.plot(values.index, values, color="#195b8a", marker="o", markersize=4.5)
    ax.axvline(pd.Timestamp("2012-10-29"), color="#838e98", linestyle=":",
               linewidth=1.0)
    ax.set(title=title, ylabel="Rentals", ylim=(0, 6700))
axes[-1].xaxis.set_major_locator(mdates.DayLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
axes[-1].set_xlabel("Date (2012)")
plt.show()

10 月 29 日には，選んだ時間帯の行がありません。暦の全日付に合わせると，29 日が行として現れ，値は `NaN` になります。線がつながった図と切れた図では，29 日の値について読み手がどう受け取りそうかを説明してください。

`NaN` を 0 件へ置き換える前に，その時間帯の記録がない理由を調べる必要があります。本文の「元の行まで戻る」で，29 日・30 日の時間別記録と出典を確認します。

### 6. 前日からの変化を調べる

前日と比べて何件増減したかは，日ごとの件数をばらばらに見ただけでは分かりません。まず日付が一日ずつ続いているかを確かめ，`shift(1)` で前日の値を当日の隣に置きます。日付に抜けがあるまま一行ずらすと，必ずしも前日の値にはならない点に注意してください。

その後，元の日付順と，一度無作為に並べ替えた順で，隣り合う二つの件数の差を比べます。近い日の件数が似ているなら，どちらの順序で差が小さくなりそうでしょうか。

In [ ]:
all_days = pd.date_range(rentals_per_day.index.min(), rentals_per_day.index.max(), freq="D")
rentals_on_calendar = rentals_per_day.reindex(all_days)
print(len(rentals_on_calendar), rentals_on_calendar.isna().sum())

In [ ]:
previous_day = rentals_on_calendar.shift(1)
nearby = pd.DataFrame({"today": rentals_on_calendar, "yesterday": previous_day})
nearby["change"] = nearby["today"] - nearby["yesterday"]
nearby.loc["2012-10-28":"2012-10-31"]

In [ ]:
reordered = rentals_on_calendar.sample(frac=1, random_state=11)
day_gap = nearby["change"].abs()
reordered_gap = reordered.diff().abs()
week_gap = (rentals_on_calendar - rentals_on_calendar.shift(7)).abs()
print(day_gap.median(), reordered_gap.median(), week_gap.median())

前日との差の絶対値の中央値は 468.5 件，一度並べ替えた順での隣同士では 1,808.5 件です。使った一日ごとの件数は同じでも，どの二日を組にするかによって差が変わりました。元の日付順には，近い日の件数が似る傾向を調べるための情報があります。

ここでは順序を一度だけ替えて比較しました。第 10 章のように仮説のもとで並べ替えを繰り返す置換検定ではありません。本文の「時間の順序を使う」で，曜日や季節が日々の利用にどう関わり得るかも考えます。

## Jupyter 実習（約 20 分）

ここからは 6 月 27 日，7 月 4 日，7 月 11 日の三つの水曜日を調べます。同じ曜日でも，朝の利用に備える場合と夜の利用に備える場合で，参考にしたい日は変わるでしょうか。

まず日合計から予想し，時間別の図で確かめます。最後に朝と夜の時間帯をそれぞれまとめ，担当者への説明を書きます。講義の二日の結果をそのまま当てはめず，この三日で確かめてください。

In [ ]:
wednesdays = pd.to_datetime(["2012-06-27", "2012-07-04", "2012-07-11"])
three_days = hourly.loc[hourly["dteday"].isin(wednesdays)]
three_days[["dteday", "hr", "cnt", "holiday"]].head()

## 1. 一日ごとの合計から予想する（約 5 分）

次の表で，三日の全体の利用規模を比べます。`hours_in_table` は収録された時間帯の行数，`rentals` は貸出件数の合計です。`holiday` は D.C. の祝日かどうかを示します。

表を見たら，**8 時台の貸出が最も多いと思う日**を一つ選び，理由を一文で書いてください。日合計が多い日ほど朝も多いと思うか，祝日かどうかが関係しそうかなど，何を手掛かりにしたかを残します。例えば「同じ曜日なら利用が多い時間帯も似る」「祝日には朝と夜の利用が違う」という説明を考えられます。自分が考えた説明なら，朝と夜の点や日付の順位がどうなりそうかも書き，次の図と照合します。

In [ ]:
daily_summary = three_days.groupby("dteday").agg(
    hours_in_table=("hr", "size"),
    rentals=("cnt", "sum"),
    holiday=("holiday", "first"),
)
daily_summary

（ここに予想する日，考えた説明，その説明なら図がどうなりそうかを書く）

## 2. 時間帯ごとの件数で確かめる（約 7 分）

三日を同じ横軸と縦軸で描き，日合計では見えなかった利用の集中する時間帯を調べます。横軸の `8` の点は，その日の 8 時台に始まった貸出件数です。

まず 8 時台で，予想した日が最も多かったかを確かめてください。次に 21 時台で，朝と同じ日が最も多いかを比べます。図の下の表でも 8・17・21 時台の件数を読めます。説明に使いたい時間帯と件数を記録します。

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.2, 3.5), layout="constrained")
for date, color, style in zip(
    wednesdays, ["#195b8a", "#a7421f", "#33735d"], ["-", "--", ":"]
):
    hours_for_date = three_days.loc[three_days["dteday"] == date].sort_values("hr")
    ax.plot(hours_for_date["hr"], hours_for_date["cnt"], color=color, linestyle=style,
            marker=".", label=date.strftime("%m-%d"))
ax.set(xlabel="Clock hour", ylabel="Rentals per hour", ylim=(0, 950),
       xlim=(-0.3, 23.3), xticks=[0, 4, 8, 12, 16, 20, 23])
ax.legend(frameon=False, loc="upper left")
plt.show()

In [ ]:
three_days.loc[three_days["hr"].isin([8, 17, 21]), ["dteday", "hr", "cnt"]]

## 3. 朝と夜のまとまった時間帯を比べる（約 8 分）

担当者が数時間続けて対応する場面を考えます。まず 8〜10 時台をまとめた貸出合計を求めます。次に，セルの最初の二つの値を自分で変更し，20〜22 時台の合計を求めてください。`between()` は両端を含むので，どちらも 3 時間帯分を足します。

表と図では三つの日を同じ順で並べています。朝と夜のそれぞれで，どの日の合計が最も多かったかを件数と一緒に記録してください。朝の結果は，設定を夜へ替える前に書き留めておきます。8 時台だけを比べた場合と，8〜10 時台を合計した場合でも，最も多い日が同じかを確かめます。

最初の日合計からの予想に戻り，朝の対応を考える担当者に向けて 2〜3 文で説明します。日合計，朝の時間帯の合計，夜との違いのうち，判断に必要な数値を選んでください。最初に考えた説明に合った点と，直す必要があった点を，それらの結果に結び付けます。予想と同じ結果だった場合も，どの点や件数がその根拠かを示してください。

In [ ]:
first_hour = 8
last_hour = 10

in_window = three_days["hr"].between(first_hour, last_hour)
window_totals = three_days.loc[in_window].groupby("dteday")["cnt"].sum().reindex(wednesdays)
window_totals.name = "rentals"
window_totals

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5.8, 3.0), layout="constrained")
ax.bar(window_totals.index.strftime("%m-%d"), window_totals,
       color=["#195b8a", "#a7421f", "#33735d"])
ax.set(xlabel="Date (2012)", ylabel="Rentals in selected hours",
       title=f"Clock hours {first_hour:02d}–{last_hour:02d}",
       ylim=(0, max(1800, 1.08 * window_totals.max())))
plt.show()

（ここに朝と夜で最も貸出が多かった日と件数，担当者への説明を書く）

### 理由を調べる（任意）

7 月 4 日は独立記念日です。国立公園局に，当日の夜の花火の記録があります。夜の貸出の内訳も調べたい場合は，集計する列を `cnt` から `registered` または `casual` へ替え，同じ時間帯で三日を比べます。区分によって最も多い日が変わるかを確認してください。行事の記録だけで貸出が増えた原因を決めず，利用した場所や目的など，理由を調べるために必要な資料を一つ挙げます。 [D.C. 人事局の 2012 年祝日表](https://dchr.dc.gov/sites/default/files/dc/sites/dchr/publication/attachments/DCHR_edpm_12_261_schedules_holidays_2012_2013.pdf) ・ [米国国立公園局の花火の記録](https://www.nps.gov/media/photo/gallery-item.htm?gid=AD5299B3-1DD8-B71B-0BADBF711BBDD03B&id=ad800bf9-1dd8-b71b-0b28-ff00da238951)

提出する場合はカーネルを再起動し，上から全セルを実行して，選んだ時間帯の図と説明が残る状態で保存します。

### 出典

H. Fanaee-T, *Bike Sharing*, UCI Machine Learning Repository（CC BY 4.0）。Capital Bikeshare の 2011〜2012 年の実測ログを研究者が集計した時間別の表を使いました。CSV は原配布の `hour.csv` の名前と改行コードを変更したものです。 [データ DOI](https://doi.org/10.24432/C5W894) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)